# Industry DSR Impact Analysis

**Apples-to-apples comparison** of two model runs with identical temporal industrial demand profiles:

- **Reference**: Temporal industrial demand, NO DSR (`EU_test_run_temporal_no_dsr`)
- **DSR v3**: Temporal industrial demand + Industry DSR (`EU_test_run_dsr_v3_optional`)

Both runs use: 33 EU countries, 38 clusters, 2030 horizon, full year 2019 (2h resolution).

**Key metrics**: System cost, electricity prices, generation capacity, renewable curtailment, DSR utilization.

In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Load networks
print('Loading Reference (temporal, no DSR)...')
n_ref = pypsa.Network('results/EU_test_run_temporal_no_dsr/networks/base_s_38___2030.nc')
print('Loading DSR v3 (temporal + DSR)...')
n_dsr = pypsa.Network('results/EU_test_run_dsr_v3_optional/networks/base_s_38___2030.nc')
print('Done.')

## 1. System Cost (Objective Value)

In [ ]:
obj_ref = n_ref.objective
obj_dsr = n_dsr.objective
diff = obj_dsr - obj_ref
diff_pct = diff / obj_ref * 100

print(f'Reference (no DSR): {obj_ref/1e9:.2f} billion EUR')
print(f'DSR v3:             {obj_dsr/1e9:.2f} billion EUR')
print(f'DSR impact:         {diff/1e9:+.2f} billion EUR ({diff_pct:+.2f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Reference\n(no DSR)', 'With DSR v3'], [obj_ref/1e9, obj_dsr/1e9],
              color=['#4a90d9', '#27ae60'], edgecolor='black', linewidth=0.5)
ax.set_ylabel('Total System Cost (billion EUR/year)')
ax.set_title('System Cost Comparison')
for bar, val in zip(bars, [obj_ref/1e9, obj_dsr/1e9]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(bottom=min(obj_ref, obj_dsr)/1e9 * 0.995)
plt.tight_layout()
plt.show()

## 2. Electricity Prices

In [ ]:
# Get low voltage bus prices
lv_ref = [b for b in n_ref.buses.index if 'low voltage' in b]
lv_dsr = [b for b in n_dsr.buses.index if 'low voltage' in b]

prices_ref = n_ref.buses_t.marginal_price[lv_ref]
prices_dsr = n_dsr.buses_t.marginal_price[lv_dsr]

avg_ref = prices_ref.mean().mean()
avg_dsr = prices_dsr.mean().mean()
print(f'Average LV price (Reference): {avg_ref:.2f} EUR/MWh')
print(f'Average LV price (DSR v3):    {avg_dsr:.2f} EUR/MWh')
print(f'DSR impact: {(avg_dsr-avg_ref)/avg_ref*100:+.2f}%')

# System-wide hourly average price
hourly_ref = prices_ref.mean(axis=1)
hourly_dsr = prices_dsr.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price duration curve
ax = axes[0]
sorted_ref = hourly_ref.sort_values(ascending=False).reset_index(drop=True)
sorted_dsr = hourly_dsr.sort_values(ascending=False).reset_index(drop=True)
ax.plot(sorted_ref.values, label='Reference (no DSR)', color='#4a90d9', linewidth=1.5)
ax.plot(sorted_dsr.values, label='With DSR v3', color='#27ae60', linewidth=1.5, linestyle='--')
ax.set_xlabel('Hours (sorted)')
ax.set_ylabel('EUR/MWh')
ax.set_title('Price Duration Curve')
ax.legend()
ax.set_ylim(bottom=0)

# Price histogram
ax = axes[1]
bins = np.linspace(0, max(hourly_ref.quantile(0.99), hourly_dsr.quantile(0.99)), 50)
ax.hist(hourly_ref, bins=bins, alpha=0.6, label='Reference (no DSR)', color='#4a90d9')
ax.hist(hourly_dsr, bins=bins, alpha=0.6, label='With DSR v3', color='#27ae60')
ax.axvline(avg_ref, color='#4a90d9', linestyle='--', linewidth=1.5, label=f'Ref mean: {avg_ref:.0f}')
ax.axvline(avg_dsr, color='#27ae60', linestyle='--', linewidth=1.5, label=f'DSR mean: {avg_dsr:.0f}')
ax.set_xlabel('EUR/MWh')
ax.set_ylabel('Hours')
ax.set_title('Price Distribution')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3. Renewable Curtailment

In [ ]:
re_carriers = ['solar', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float']

results = {}
for label, n_ in [('Reference', n_ref), ('DSR v3', n_dsr)]:
    re = n_.generators[n_.generators.carrier.isin(re_carriers)]
    potential = (n_.generators_t.p_max_pu[re.index] * re.p_nom_opt).sum().sum()
    actual = n_.generators_t.p[re.index].sum().sum()
    curtailed = potential - actual
    results[label] = {'potential': potential/1e6, 'actual': actual/1e6, 'curtailed': curtailed/1e6,
                      'pct': curtailed/potential*100}
    print(f'{label}: Curtailed {curtailed/1e6:.2f} TWh ({curtailed/potential*100:.2f}%)')

curt_reduction = (results['DSR v3']['curtailed'] - results['Reference']['curtailed']) / results['Reference']['curtailed'] * 100
print(f'\nDSR reduces curtailment by {-curt_reduction:.0f}%')

# Curtailment by carrier
fig, ax = plt.subplots(figsize=(8, 5))
carrier_curt = {}
for label, n_ in [('Reference', n_ref), ('DSR v3', n_dsr)]:
    re = n_.generators[n_.generators.carrier.isin(re_carriers)]
    for c in re_carriers:
        gens_c = re[re.carrier == c]
        if gens_c.empty:
            continue
        pot = (n_.generators_t.p_max_pu[gens_c.index] * gens_c.p_nom_opt).sum().sum()
        act = n_.generators_t.p[gens_c.index].sum().sum()
        carrier_curt.setdefault(c, {})[label] = (pot - act) / 1e6

df_curt = pd.DataFrame(carrier_curt).T
df_curt.plot(kind='bar', ax=ax, color=['#4a90d9', '#27ae60'], edgecolor='black', linewidth=0.5)
ax.set_ylabel('Curtailment (TWh)')
ax.set_title('Renewable Curtailment by Carrier')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. DSR Store Utilization

In [ ]:
dsr_stores = n_dsr.stores[n_dsr.stores.carrier == 'industry dsr']

print(f'Total DSR stores: {len(dsr_stores)}')
print(f'Stores with capacity built: {(dsr_stores.e_nom_opt > 0).sum()}')
print(f'Total capacity available: {dsr_stores.e_nom_max.sum():.0f} MWh')
print(f'Total capacity built: {dsr_stores.e_nom_opt.sum():.0f} MWh ({dsr_stores.e_nom_opt.sum()/dsr_stores.e_nom_max.sum()*100:.1f}%)')
print(f'Capital cost: {dsr_stores.capital_cost.iloc[0]:.1f} EUR/MWh/year')
print(f'Total DSR investment: {(dsr_stores.e_nom_opt * dsr_stores.capital_cost).sum()/1e6:.2f} M EUR/year')

# Store capacity by technology
dsr_stores_built = dsr_stores[dsr_stores.e_nom_opt > 0].copy()
dsr_stores_built['technology'] = dsr_stores_built.index.map(
    lambda x: x.split(' industry dsr ')[-1] if ' industry dsr ' in x else 'unknown'
)

tech_cap = dsr_stores_built.groupby('technology')[['e_nom_opt', 'e_nom_max']].sum()
tech_cap = tech_cap.sort_values('e_nom_opt', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
tech_cap.plot(kind='barh', ax=ax, color=['#27ae60', '#cccccc'], edgecolor='black', linewidth=0.5)
ax.set_xlabel('Energy Capacity (MWh)')
ax.set_title('DSR Store Capacity by Technology')
ax.legend(['Built (e_nom_opt)', 'Available (e_nom_max)'])
plt.tight_layout()
plt.show()

## 5. DSR Dispatch Profile

In [ ]:
# Charge and discharge link flows
ch_links = n_dsr.links[n_dsr.links.carrier == 'industry dsr charge']
dch_links = n_dsr.links[n_dsr.links.carrier == 'industry dsr discharge']

charge_total = n_dsr.links_t.p0[ch_links.index].sum(axis=1)
discharge_total = n_dsr.links_t.p0[dch_links.index].sum(axis=1)
net_dsr = charge_total - discharge_total  # positive = extra demand, negative = reduced demand

print(f'Total energy charged (increased demand): {charge_total.sum()/1e6:.2f} TWh')
print(f'Total energy discharged (reduced demand): {discharge_total.sum()/1e6:.2f} TWh')
print(f'Efficiency loss: {(charge_total.sum()-discharge_total.sum())/1e6:.2f} TWh')
print(f'Peak charge power: {charge_total.max()/1e3:.1f} GW')
print(f'Peak discharge power: {discharge_total.max()/1e3:.1f} GW')

# Weekly average dispatch (for readability)
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Full year - weekly average
ax = axes[0]
weekly_ch = charge_total.resample('W').mean()
weekly_dch = discharge_total.resample('W').mean()
ax.fill_between(weekly_ch.index, 0, weekly_ch.values/1e3, alpha=0.6, color='#e74c3c', label='Charge (increased demand)')
ax.fill_between(weekly_dch.index, 0, -weekly_dch.values/1e3, alpha=0.6, color='#27ae60', label='Discharge (reduced demand)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Power (GW)')
ax.set_title('DSR Dispatch - Weekly Average')
ax.legend()

# Typical week - hourly
ax = axes[1]
jan_start = charge_total.index[0] + pd.Timedelta(days=7)
jan_end = jan_start + pd.Timedelta(days=7)
ch_week = charge_total[jan_start:jan_end]
dch_week = discharge_total[jan_start:jan_end]
ax.fill_between(ch_week.index, 0, ch_week.values/1e3, alpha=0.6, color='#e74c3c', label='Charge')
ax.fill_between(dch_week.index, 0, -dch_week.values/1e3, alpha=0.6, color='#27ae60', label='Discharge')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Power (GW)')
ax.set_xlabel('Time')
ax.set_title('DSR Dispatch - Sample Week (January)')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Price Impact of DSR by Hour of Day

In [ ]:
# Average price by hour of day
hourly_ref = prices_ref.mean(axis=1)
hourly_dsr_price = prices_dsr.mean(axis=1)

by_hour_ref = hourly_ref.groupby(hourly_ref.index.hour).mean()
by_hour_dsr = hourly_dsr_price.groupby(hourly_dsr_price.index.hour).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(by_hour_ref.index, by_hour_ref.values, 'o-', label='Reference (no DSR)', color='#4a90d9', linewidth=2)
ax.plot(by_hour_dsr.index, by_hour_dsr.values, 's--', label='With DSR v3', color='#27ae60', linewidth=2)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Price (EUR/MWh)')
ax.set_title('Average Electricity Price by Hour of Day')
ax.legend()
ax.set_xticks(range(0, 24, 2))

# Price difference by hour
ax = axes[1]
price_diff_hour = by_hour_dsr - by_hour_ref
colors = ['#27ae60' if v < 0 else '#e74c3c' for v in price_diff_hour.values]
ax.bar(price_diff_hour.index, price_diff_hour.values, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Price Difference (EUR/MWh)')
ax.set_title('DSR Price Impact by Hour (negative = cheaper)')
ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

## 7. Generation Mix Comparison

In [ ]:
# Compare generation by carrier
gen_data = {}
for label, n_ in [('Reference', n_ref), ('DSR v3', n_dsr)]:
    gen_by_carrier = n_.generators_t.p.sum().groupby(n_.generators.carrier).sum() / 1e6  # TWh
    gen_data[label] = gen_by_carrier

df_gen = pd.DataFrame(gen_data).fillna(0)
df_gen['Difference'] = df_gen['DSR v3'] - df_gen['Reference']
df_gen = df_gen.sort_values('Reference', ascending=False)

# Top carriers
top = df_gen.head(15)
print('Generation by carrier (TWh):')
print(top.to_string(float_format='{:.1f}'.format))

# Plot difference
diff_nonzero = df_gen[df_gen['Difference'].abs() > 0.01]['Difference'].sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#27ae60' if v > 0 else '#e74c3c' for v in diff_nonzero.values]
diff_nonzero.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Generation Difference (TWh)')
ax.set_title('Change in Generation by Carrier (DSR v3 vs Reference)')
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
# Summary table
dsr_stores = n_dsr.stores[n_dsr.stores.carrier == 'industry dsr']
ch_links = n_dsr.links[n_dsr.links.carrier == 'industry dsr charge']
dch_links = n_dsr.links[n_dsr.links.carrier == 'industry dsr discharge']

re_carriers = ['solar', 'onwind', 'offwind-ac', 'offwind-dc', 'offwind-float']
curt = {}
for label, n_ in [('Reference', n_ref), ('DSR v3', n_dsr)]:
    re = n_.generators[n_.generators.carrier.isin(re_carriers)]
    pot = (n_.generators_t.p_max_pu[re.index] * re.p_nom_opt).sum().sum()
    act = n_.generators_t.p[re.index].sum().sum()
    curt[label] = (pot - act) / 1e6

summary = pd.DataFrame({
    'Metric': [
        'System Cost (B EUR/year)',
        'Average LV Price (EUR/MWh)',
        'Renewable Curtailment (TWh)',
        'DSR Capacity Built (MWh)',
        'DSR Energy Shifted (TWh)',
        'DSR Investment (M EUR/year)',
        'System Cost Savings (B EUR/year)',
    ],
    'Reference (no DSR)': [
        f'{n_ref.objective/1e9:.2f}',
        f'{avg_ref:.2f}',
        f'{curt["Reference"]:.2f}',
        'N/A',
        'N/A',
        'N/A',
        'N/A',
    ],
    'DSR v3': [
        f'{n_dsr.objective/1e9:.2f}',
        f'{avg_dsr:.2f}',
        f'{curt["DSR v3"]:.2f}',
        f'{dsr_stores.e_nom_opt.sum():.0f}',
        f'{n_dsr.links_t.p0[ch_links.index].sum().sum()/1e6:.2f}',
        f'{(dsr_stores.e_nom_opt * dsr_stores.capital_cost).sum()/1e6:.2f}',
        f'{(n_ref.objective - n_dsr.objective)/1e9:.2f}',
    ],
})

print('\n' + '='*70)
print('INDUSTRY DSR IMPACT SUMMARY')
print('='*70)
print(summary.to_string(index=False))
print('\nConclusion: DSR reduces system cost, lowers prices, and cuts curtailment.')
print('The optimizer rationally builds DSR where the arbitrage value exceeds costs.')